# Multi-service Docker Compose: a step-through

> last_verified: 2026-07-31 · docker compose

Single-container demos are fine for getting started, but real apps are multi-service.
This notebook walks through standing up a two-service stack with Docker Compose — a
Python/Flask web app and a PostgreSQL database — and exercises the commands I use to
inspect, verify, and debug the running stack.

Everything here uses the Compose v2 CLI (`docker compose`, not the legacy
`docker-compose` command). Docker >= 20.10 ships the Compose v2 plugin we use
here [source](https://markaicode.com/integrate/kubernetes-with-docker/).

## What I'm building

- **web** — a small Flask app that reads a table count from PostgreSQL and serves it on port 5000.
- **db** — PostgreSQL with a healthcheck so Compose knows when it's truly ready.

Compose auto-creates a single-user-defined bridge network and gives each service a
DNS name matching its service key. That means `web` can reach the database at the
hostname `db` without any extra networking configuration.

In [ ]:
%%bash

# Create a clean project directory
PROJECT_DIR=/tmp/compose-demo
rm -rf "$PROJECT_DIR"
mkdir -p "$PROJECT_DIR/app"
cd "$PROJECT_DIR"
echo "Working in $(pwd)"

In [ ]:
%%bash
cat > docker-compose.yml <<'YAML'
services:
  web:
    image: python:3.11-slim
    working_dir: /app
    volumes:
      - ./app:/app
      - ./requirements.txt:/app/requirements.txt:ro
    ports:
      - "5000:5000"
    environment:
      DATABASE_URL=postgresql://postgres:postgres@db:5432/appdb
    depends_on:
      db:
        condition: service_healthy
    command: bash -c "pip install -r requirements.txt && python app.py"

  db:
    image: postgres:15-alpine
    environment:
      POSTGRES_USER: postgres
      POSTGRES_PASSWORD: postgres
      POSTGRES_DB: appdb
    volumes:
      - db-data:/var/lib/postgresql/data
    healthcheck:
      test: ["CMD-SHELL", "pg_isready -U postgres"]
      interval: 5s
      timeout: 5s
      retries: 5

volumes:
  db-data:
YAML

cat > app/requirements.txt <<'REQ'
flask==3.0.3
psycopg2-binary==2.9.9
REQ

cat > app/app.py <<'PY'
import os, psycopg2
from flask import Flask

app = Flask(__name__)
DB_URL = os.environ["DATABASE_URL"]

@app.route("/")
def index():
    conn = psycopg2.connect(DB_URL, connect_timeout=3)
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM pg_tables WHERE schemaname='public'")
    tables = cur.fetchone()[0]
    cur.close()
    conn.close()
    return f"Flask app running. Public tables in DB: {tables}\n"

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)
PY

echo "Files created:"
ls -R "$PROJECT_DIR"

## Validate before starting

I run `docker compose config` first — it parses the YAML, resolves the full
configuration, and reports any errors without starting containers. This catches
typos in service names and volume references before I waste time pulling images.

In [ ]:
%%bash
cd /tmp/compose-demo
docker compose config 2>&1 | head -40

## Start the stack and verify

Now I bring everything up in the background with `-d`. Compose pulls the images,
creates the volume, sets up the bridge network, and starts both containers.
The `depends_on` with `condition: service_healthy` means Compose waits until the
PostgreSQL healthcheck passes before starting the web service — something I learned
the hard way after too many 'connection refused' errors.

In [ ]:
%%bash
cd /tmp/compose-demo
docker compose up -d
echo "---"
docker compose ps

## Test the endpoint and read the logs

If the app is healthy, `curl localhost:5000` should return the table count from
PostgreSQL. I also tail the logs to see any runtime errors.

In [ ]:
%%bash
cd /tmp/compose-demo
sleep 5  # give the web app a moment to start after the db became healthy
curl -sf localhost:5000 || echo "(curl failed — check logs below)"
echo "--- web logs ---"
docker compose logs --tail=20 web

## Inspect the auto-created network

Compose created a network named after the project directory. Both services share it
and can reach each other by service name. Let me confirm the network exists and
list the containers attached to it.

In [ ]:
%%bash
cd /tmp/compose-demo
docker network ls --filter name=compose-demo_default
echo "---"
docker inspect compose-demo_default 2>/dev/null \
  --format '{{range .Containers}}{{.Name}} -> {{.IPv4Address}}{{end}}'

## What I got stuck on

- **Service names as hostnames:** I initially tried connecting to `localhost:5432`
  from the web container. That doesn't work — each container has its own network
  namespace. Within Compose, the database is reachable at `db:5432`.
- **depends_on alone isn't a readiness check:** `depends_on` only waits for the
  container to start, not for the process inside to be ready. The healthcheck +
  `condition: service_healthy` pairing is what actually synchronizes startup.
- **Named volume lifecycle:** the `db-data` volume persists data between
  `docker compose down` runs. To start fresh I use `docker compose down -v`.

## What I'd try next

I want to take this multi-service setup and explore three directions: add more web
replicas with `docker compose up --scale web=3` and test load balancing, add a
healthcheck to the web service for orchestration platforms, and investigate the
bridge to Kubernetes when the app outgrows a single host
[source](https://markaicode.com/integrate/kubernetes-with-docker/).